In [ ]:
# FEATURE IMPORTANCE REPORT
Heart Disease Prediction | Technical Case Study

Purpose: rank feature contributions using multiple interpretability techniques, compare global and local importance, and justify feature selection. Populate result tables using actual notebook outputs; this report does not invent feature rankings or performance values.

# 1. Executive Summary

Feature importance is not one universal quantity. Random Forest impurity importance measures split-based contribution; permutation importance measures the effect of disrupting a feature on a selected score; SHAP estimates feature contributions to individual predictions relative to background data. These methods answer different questions, so their rankings should be compared rather than treated as interchangeable.

Feature selection should consider predictive usefulness, rank stability, clinical plausibility, data quality, redundancy, fairness, and availability at prediction time—not importance scores alone.

# 2. Dataset and Model

Dataset: heart.csv | Target: HeartDisease (0 = no heart disease; 1 = heart disease) | Model: Random Forest Classifier. The code assumes `preprocessor`, `model`, `X_train_processed`, `X_test_processed`, `y_test` are already defined. Fit preprocessing only on training data. Use validation or cross-validation for feature selection; reserve the test set for final evaluation.

# 3. Interpretability Methods

# Technique

# Scope

Measures

Limitation

RF impurity importance

Global

Impurity reduction assigned to tree splits

Can favour high-cardinality features; correlated variables split importance.

# Permutation importance

Global

Change in selected score after shuffling a feature

Correlated features may mask each other; depends on metric.

# Mean absolute SHAP

Global summary

Average magnitude of feature attributions

Depends on background data and feature-dependence assumptions.

# SHAP for one case

Local

Direction and size of contributions to one prediction

One case is not representative of the whole population.

# 4. Reproducible Notebook Code

4.1 Random Forest importance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

feature_names = preprocessor.get_feature_names_out()

rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "RF_Importance": model.feature_importances_
})
rf_importance["RF_Rank"] = rf_importance["RF_Importance"].rank(
    ascending=False, method="average"
)

4.2 Permutation importance

from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model, X_test_processed, y_test,
    n_repeats=20, random_state=42, scoring="f1"
)
perm_df = pd.DataFrame({
    "Feature": feature_names,
    "Permutation_Mean": perm.importances_mean,
    "Permutation_Std": perm.importances_std
})
perm_df["Permutation_Rank"] = perm_df["Permutation_Mean"].rank(
    ascending=False, method="average"
)

4.3 SHAP global and local importance

%pip install shap

import shap
background = X_train_processed.sample(
    n=min(100, len(X_train_processed)), random_state=42
)
explainer = shap.TreeExplainer(model, data=background)
X_eval = X_test_processed.iloc[:min(100, len(X_test_processed))]
shap_exp = explainer(X_eval)

values = shap_exp.values
# Binary classification: select positive class if output is 3-dimensional.
positive_values = values[:, :, 1] if values.ndim == 3 else values

shap_global = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_MeanAbs": np.abs(positive_values).mean(axis=0)
})
shap_global["SHAP_Global_Rank"] = shap_global["SHAP_MeanAbs"].rank(
    ascending=False, method="average"
)

local_index = 0
shap_local = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_Local": positive_values[local_index]
})
shap_local["Absolute_Impact"] = shap_local["SHAP_Local"].abs()
shap_local = shap_local.sort_values("Absolute_Impact", ascending=False)
print(shap_local.head(15))

4.4 Manual consensus ranking

ranked = (rf_importance
    .merge(perm_df, on="Feature")
    .merge(shap_global, on="Feature"))

rank_cols = ["RF_Rank", "Permutation_Rank", "SHAP_Global_Rank"]
ranked["Mean_Rank"] = ranked[rank_cols].mean(axis=1)
ranked["Consensus_Rank"] = ranked["Mean_Rank"].rank(
    ascending=True, method="min"
).astype(int)

ranked = ranked.sort_values("Consensus_Rank")
print(ranked)
ranked.to_csv("feature_importance_rankings.csv", index=False)

Ranking rule: each method ranks features from 1 (most important) downward. The consensus score is the arithmetic mean of the three ranks; a lower mean rank indicates stronger agreement. Equal weighting is a transparent descriptive choice, not a statistically validated importance score. Always show the individual method rankings so disagreements remain visible.

5. Global vs Local Comparison

Global importance summarizes behaviour across many examples. Local importance explains one selected prediction. A feature can rank highly globally but contribute little to one patient prediction; another feature with modest global importance can strongly influence a particular case.

# Global consensus plot
plot_df = ranked.head(15).sort_values("Mean_Rank", ascending=False)
sns.barplot(data=plot_df, x="Mean_Rank", y="Feature")
plt.title("Global Consensus Ranking (lower rank is better)")
plt.tight_layout()
plt.show()

# Local SHAP plot for the selected case
local_plot = shap_local.head(15).sort_values("SHAP_Local")
sns.barplot(data=local_plot, x="SHAP_Local", y="Feature")
plt.axvline(0, color="black")
plt.title("Local SHAP Contributions")
plt.tight_layout()
plt.show()

6. Results Tables to Complete

Table A. Global feature rankings — copy actual results from `ranked`.

Feature

RF rank

Permutation rank

SHAP global rank

Consensus rank

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

Table B. Local explanation — copy actual results from `shap_local`.

Feature

Local SHAP value

Direction

Interpretation

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

[Insert actual result]

# 7. Technical Feature Selection Justification

Prioritize features that rank consistently across multiple methods; do not rely on one score.

Investigate large ranking disagreements. They may arise from correlation, cardinality effects, interactions, or differences between global and local explanations.

Use validation-based permutation importance and clinically relevant metrics. For imbalanced outcomes, consider recall, F1, or PR-AUC in addition to accuracy.

Check ranking stability across cross-validation folds or bootstrap samples. Treat unstable rankings cautiously.

Review missingness, measurement reliability, leakage risk, availability at prediction time, and potential bias.

Ask qualified clinical experts whether selected variables are meaningful and whether important context is missing.

Prefer a smaller feature set only if validation performance, calibration, stability, and fairness remain acceptable; evaluate the final pipeline on a separate test set.

# 8. Technical Limitations

Feature importance is model-dependent and does not establish causation.

Correlated predictors can mask one another in permutation importance.

Random Forest impurity importance can be biased toward features with many possible split points or categories.

SHAP depends on the background dataset and feature-dependence assumptions.

One-hot encoded categories may need to be grouped back into their original feature for clinical reporting.

A local explanation for one patient must not be generalized to all patients.

Predicted probabilities should not be described as actual clinical risk unless validated and calibrated for the intended population.

# 9. Feature Selection Decision Framework

# Criterion

Evidence to collect

Decision use

Predictive usefulness

Cross-validated score and uncertainty

Retain only if it adds reliable predictive value.

# Interpretability consistency

Rank agreement across RF, permutation, SHAP

Investigate disagreements; favour consistent evidence.

# Stability

Rank variation across folds or resamples

Avoid overvaluing unstable features.

# Clinical validity

Qualified expert review and relevant evidence

Check plausibility and missing context.

Data quality and leakage

Missingness, measurement, prediction-time availability

Exclude leaked or unreliable features.

# Fairness and safety

Subgroup performance and error analysis

Mitigate harmful disparities before deployment.

# 10. Conclusion

A defensible feature-importance report combines multiple global techniques with local explanations. Random Forest impurity importance, permutation importance, and SHAP provide complementary views, not interchangeable scores. A consensus ranking can organize evidence, but final feature selection must also consider validation performance, stability, clinical review, data quality, fairness, and deployment constraints. Complete the rankings and conclusions using actual notebook outputs.

Educational-use disclaimer: This Heart Disease Prediction project is not a clinically validated diagnostic system and must not be used independently to make medical decisions.

Feature Importance Report | Heart Disease Prediction

